In [280]:
import pandas as pd
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

In [281]:
#DATA CLEANING

In [282]:
locals_df = pd.read_excel(r"C:\Users\javie\Downloads\Locals Madrid data.xlsx")

In [283]:
locals_df.head()

,Name,Type,Rating,Reviews,WEB,Online service
0,Carmen Blanco,Retail,4.2,9,NO,NO
1,Arreglo de ropa,Retail,4.3,16,NO,NO
2,Alex y Lola,Restorant,4.9,120,NO,NO
3,El Llagar de Maiquez,Restorant,4.0,1159,NO,NO
4,Comercial Ibiza,Retail,4.0,29,NO,NO


In [284]:
locals_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            62 non-null     object 
 1   Type            62 non-null     object 
 2   Rating          62 non-null     float64
 3   Reviews         62 non-null     int64  
 4   WEB             62 non-null     object 
 5   Online service  62 non-null     object 
dtypes: float64(1), int64(1), object(4)
memory usage: 3.0+ KB


In [285]:
locals_df.describe()

,Rating,Reviews
count,62.000000,62.000000
mean,4.488710,705.225806
std,0.412153,1303.321751
min,2.700000,1.000000
25%,4.225000,26.250000
50%,4.600000,185.000000
75%,4.800000,725.500000
max,5.000000,6434.000000


In [286]:
locals_df.sample(10)

,Name,Type,Rating,Reviews,WEB,Online service
18,Esperanza,Barber,4.8,5,NO,NO
50,erre que erre,Restorant,4.7,2260,YES,NO
6,La fragua de vito,Restorant,4.5,732,NO,NO
34,Sidreria Carlos tarterie,restorant,4.2,1536,YES,YES
22,El espin,Restorant,4.4,24,NO,NO
14,Kime,Barber,4.3,10,NO,NO
56,donpiso,retail,4.8,174,YES,NO
29,Old style barber,Barber,4.7,227,NO,NO
44,la retasca,restorant,4.0,1694,YES,YES
30,Asiyas coffe,Restorant,4.7,27,NO,NO


In [287]:
locals_df["WEB"] = locals_df["WEB"].map({"YES": 1, "yes":1, "NO":0, "no":0})
locals_df["Online service"] = locals_df["Online service"].map({"YES":1, "yes":1, "NO":0, "no":0})

In [288]:
locals_df.isnull().sum()

Name              0
Type              0
Rating            0
Reviews           0
WEB               0
Online service    0
dtype: int64

In [289]:
#EDA

In [290]:
web = locals_df["WEB"].value_counts(normalize=True)
online = locals_df["Online service"].value_counts(normalize=True)
print(web, online)

WEB
0    0.5
1    0.5
Name: proportion, dtype: float64 Online service
0    0.677419
1    0.322581
Name: proportion, dtype: float64


In [291]:
# 50% has web, 50% not have
# 68% not have Online service, 32% have

In [292]:
#Rating mean vs WEB 
web_rating = locals_df[locals_df["WEB"] == 0]["Rating"]
non_web_rating = locals_df[locals_df["WEB"] == 1]["Rating"]
rating_mean = locals_df.groupby("WEB")["Rating"].mean()
rating_mean

WEB
0    4.43871
1    4.53871
Name: Rating, dtype: float64

In [293]:
# Locals with web has very low difference of better rating than the ones without WEB

In [294]:
# Because we dont know if the difference of rating mean is high or not, we make comparison between every local rating and the mean of total ratings
total_mean = locals_df["Rating"].mean()
total_mean

np.float64(4.488709677419355)

In [295]:
locals_df["Mean_Distribution"] = locals_df["Rating"] - total_mean
locals_df[["Name", "Rating", "Mean_Distribution"]].sample(10)

,Name,Rating,Mean_Distribution
52,rafa,4.4,-0.08871
56,donpiso,4.8,0.31129
57,laboxdeguita,4.9,0.41129
55,txirimiri,4.1,-0.38871
47,spa milagros,4.9,0.41129
5,Moka Home,4.1,-0.38871
25,Linea,4.9,0.41129
34,Sidreria Carlos tarterie,4.2,-0.28871
54,cecilia baron,4.9,0.41129
27,Casa Gardel,4.7,0.21129


In [296]:
# Locals above the mean
locals_df["above_mean"] = (locals_df["Mean_Distribution"] > 0).astype(int)
locals_df["above_mean"].value_counts()

above_mean
1    36
0    26
Name: count, dtype: int64

In [297]:
#Locals without WEB and rating over the mean --> Target for businness
premium_locals = locals_df[(locals_df["WEB"] == 0) & (locals_df["Mean_Distribution"] > 0)]
premium_locals.sample(10)

,Name,Type,Rating,Reviews,WEB,Online service,Mean_Distribution,above_mean
30,Asiyas coffe,Restorant,4.7,27,0,0,0.21129,1
2,Alex y Lola,Restorant,4.9,120,0,0,0.41129,1
6,La fragua de vito,Restorant,4.5,732,0,0,0.01129,1
8,Electricidad Vife,Retail,4.6,17,0,0,0.11129,1
17,El estribo,Pub,4.6,474,0,0,0.11129,1
10,Sirtaki,Restorant,4.9,73,0,0,0.41129,1
15,El raton vaquero,Restorant,5.0,1,0,0,0.51129,1
21,Joyeria,Retail,5.0,3,0,0,0.51129,1
26,Il Giovale,Restorant,4.9,510,0,0,0.41129,1
11,Juani,Barber,4.8,6,0,0,0.31129,1


In [317]:
premium_locals.count()

Name                 17
Type                 17
Rating               17
Reviews              17
WEB                  17
Online service       17
Mean_Distribution    17
above_mean           17
dtype: int64

In [298]:
#Reviews mean vs WEB
web_reviews = locals_df[locals_df["WEB"] == 0]["Reviews"]
non_web_reviews = locals_df[locals_df["WEB"] == 1]["Reviews"]
reviews_mean = locals_df.groupby("WEB")["Reviews"].mean()
reviews_mean

WEB
0     224.870968
1    1185.580645
Name: Reviews, dtype: float64

In [299]:
# Locals with WEB has much more reviews than the ones that dont have web

In [300]:
# Locals with high reviews and rating
locals_df["high_reputation"] = ((locals_df["Rating"] >= 4.5) & (locals_df["Reviews"] >= 100)).astype(int)
locals_df["high_reputation"].value_counts()

high_reputation
0    43
1    19
Name: count, dtype: int64

In [301]:
# 19 locals has a high reputation (rating over 4.5 and over 100 reviews)
#See the difference in the 19 locals with and without web
high_reputation = locals_df[locals_df["high_reputation"] == 1]["WEB"].value_counts(normalize=True)
high_reputation

WEB
1    0.578947
0    0.421053
Name: proportion, dtype: float64

In [302]:
# 58% of WEB locals are "premium"
# 42% of non-WEB locals are "premium" --> business target

In [303]:
#WEB precense in low and high reputation
goodness_of_fit = pd.crosstab(locals_df["high_reputation"], locals_df["WEB"])
percentage_gof = pd.crosstab(locals_df["high_reputation"], locals_df["WEB"], normalize="index")# Preguntar importancia de estos porcentajes
print(goodness_of_fit, percentage_gof)

WEB               0   1
high_reputation        
0                23  20
1                 8  11 WEB                     0         1
high_reputation                    
0                0.534884  0.465116
1                0.421053  0.578947


In [304]:
# Observed Table

#            WEB   0   1    TOTAL
#high_reputation        
#0                23  20    43
#1                 8  11    19
# TOTAL           31  31    62

#              WEB 0    1       TOTAL
#high_reputation                    
#0               34,5% 34,5%    69%
#1               16,5% 16,5%    31%
# Total           50%  50%      100%

# Expected Table
#            WEB   0   1    TOTAL
#high_reputation        
#0                21,5  21,5  43
#1                9,5   9,5   19
# TOTAL           31    31    62

In [305]:
table = st.chi2_contingency(goodness_of_fit)
table

Chi2ContingencyResult(statistic=np.float64(0.30354957160342716), pvalue=np.float64(0.5816656678853607), dof=1, expected_freq=array([[21.5, 21.5],
       [ 9.5,  9.5]]))

In [306]:
#HYPOTESIS TESTING
#1
# H0: \mu rating mean with web == \mu rating mean without web
# H1: \mu rating mean with web != \mu rating mean without web

#2
# H0: \mu reviews with web == \mu reviews without web
# H1: \mu reviews with web != \mu reviews without web

In [307]:
p_value_rating = st.ttest_ind(web_rating, non_web_rating, equal_var=False)
p_value_rating

TtestResult(statistic=np.float64(-0.9545317048435109), pvalue=np.float64(0.34432103062067987), df=np.float64(50.95657968943969))

In [308]:
p_value_reviews = st.ttest_ind(web_reviews, non_web_reviews, equal_var=False)
p_value_reviews

TtestResult(statistic=np.float64(-3.1001289989944136), pvalue=np.float64(0.004023185841579248), df=np.float64(31.907729176101267))

In [309]:
locals_df.groupby("WEB")["Reviews"].describe()

,count,mean,std,min,25%,50%,75%,max
WEB,,,,,,,,
0,31.0,224.870968,303.034623,1.0,12.0,70.0,372.0,1159.0
1,31.0,1185.580645,1698.594277,10.0,76.5,525.0,1305.5,6434.0


In [310]:
locals_df["log_reviews"] = np.log1p(locals_df["Reviews"])

In [311]:
reviews_no_web = locals_df[locals_df["WEB"] == 0]["log_reviews"]
reviews_web = locals_df[locals_df["WEB"] == 1]["log_reviews"]

In [312]:
p_value_reviews = st.ttest_ind(reviews_web, reviews_no_web, equal_var=False)
p_value_reviews

TtestResult(statistic=np.float64(3.795346120451294), pvalue=np.float64(0.0003458915159644135), df=np.float64(59.944038678791514))

In [313]:
Locals_Dataframe = locals_df.to_excel(r"C:\Users\javie\Desktop\source_for_tableau.xlsx", index=False)

In [314]:
import os
os.getcwd()

'c:\\Users\\javie\\AppData\\Local\\Programs\\Microsoft VS Code'